## Read Order Data from JSON

In [0]:
#/Volumes/workspace/default/orderdata/Order.json

orderDF = spark.read.option("multiline", "true").json("/Volumes/workspace/default/orderdata/Order.json")
#updatedorderDF = spark.read.option("multiline", "true").json("/Volumes/workspace/default/orderdata/UpdatedOrder.json")
    

In [0]:
display(orderDF)

In [0]:
updatedorderDF = spark.read.option("multiline", "true").json("/Volumes/workspace/default/orderdata/UpdatedOrder.json")
display(updatedorderDF)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

#Flatten array of structs and structs
def flatten(df):
   #compute Complex Fields (Lists and Structs) in Schema   
   complex_fields = dict([(field.name, field.dataType)
                             for field in df.schema.fields
                             if type(field.dataType) == ArrayType or  type(field.dataType) == StructType])
   
   while len(complex_fields)!=0:
      col_name=list(complex_fields.keys())[0]
      print ("Processing :"+col_name+" Type : "+str(type(complex_fields[col_name])))
    
      #if StructType then convert all sub element to columns.
      #i.e. flatten structs
      if (type(complex_fields[col_name]) == StructType):
         expanded = [col(col_name+'.'+k).alias(col_name+'_'+k) for k in [ n.name for n in  complex_fields[col_name]]]
         df=df.select("*", *expanded).drop(col_name)
         print ("flattening structs :"+col_name+" Type : "+str(type(complex_fields[col_name])))
         display(df)

      #if ArrayType then add the Array Elements as Rows using the explode function
      #i.e. explode Arrays
      elif (type(complex_fields[col_name]) == ArrayType):    
         df=df.withColumn(col_name,explode_outer(col_name))
         print("xplode function")
    
      #recompute remaining Complex Fields in Schema       
      complex_fields = dict([(field.name, field.dataType)
                             for field in df.schema.fields
                             if type(field.dataType) == ArrayType or  type(field.dataType) == StructType])
      print("Last step of function")
   return df

In [0]:
flatOrderData = flatten(orderDF)
display(flatOrderData)